In [85]:
import utils.data_loader as data_loader
import numpy as np
import pandas as pd
import os
import scipy
from pathlib import Path

WindowsPath('c:/Users/abhyu/Documents/SINTEF')

In [59]:
x,y,z = data_loader.load_raw_data('data/raw_data/', filter3D=pd.concat([pd.read_csv('data/nodes/c1.csv'),pd.read_csv('data/nodes/c2.csv'),pd.read_csv('data/nodes/sus.csv')]))

In [83]:
data = {}
for k in x.keys():
    if k == 'columns':
        continue
    name = k.split('_')[0]
    res_k = ''.join([name, '_Resultsabridged.csv'])
    data[name] = {'temps': x[k], 'res': z[res_k]}


In [84]:
data

{'PW10': {'temps': array([[7.17336352e-01, 3.13718758e-02, 0.00000000e+00, ...,
          3.08359181e+02, 2.96518098e+02, 3.58300000e+03],
         [7.50000000e-01, 0.00000000e+00, 0.00000000e+00, ...,
          3.07434241e+02, 2.95628909e+02, 3.58400000e+03],
         [7.44086026e-01, 9.39999252e-02, 0.00000000e+00, ...,
          3.07323564e+02, 2.95520148e+02, 3.59500000e+03],
         ...,
         [8.33305571e-02, 6.05431936e-02, 1.20000000e+00, ...,
          3.38560902e+02, 3.25438719e+02, 1.75620000e+04],
         [1.19638689e-01, 4.60705477e-02, 1.20000000e+00, ...,
          3.38075142e+02, 3.24968534e+02, 1.75640000e+04],
         [1.12006292e-01, 8.13773344e-02, 1.20000000e+00, ...,
          3.37836853e+02, 3.24738163e+02, 1.75660000e+04]], shape=(4038, 65)),
  'res': array([[0.00000000e+00, 4.66526509e+02, 2.79915905e-05, ...,
          2.00000000e+01, 2.00000000e+01, 2.00000000e+01],
         [5.00000000e-01, 4.66526509e+02, 2.43706928e-01, ...,
          2.04438684e+01,

In [69]:
U = scipy.linalg.svd(all)[0]

In [72]:
Ur = U[:10,:]
Ur.shape

(10, 4038)

In [13]:
nodes = x['PW10_T3Dabridged.csv'][:,:3]
df = pd.DataFrame(nodes, columns=['X', 'Y', 'Z'])
df

,X,Y,Z
0,1.199769,0.023526,-0.325000
1,1.199075,0.047112,-0.350000
2,1.200000,0.000000,-0.350000
3,1.168118,0.031043,-0.350000
4,1.176678,0.000000,-0.325000
...,...,...,...
21037,0.038841,0.000000,1.474049
21038,0.032057,0.023290,1.475140
21039,0.013462,0.009781,1.481399
21040,0.016275,0.000000,1.481164


In [35]:
c1 = df[['X', 'Y', 'Z']]#.to_numpy()
c1 = c1[c1['X']**2 + c1['Y']**2 <= 0.2**2]
c1 = c1[c1['Z'] <= 1.1]
c1 = c1[c1['Z'] >= 0.1].drop_duplicates() 

c2 = df[['X', 'Y', 'Z']]
c2 = c2[c2['Z'] >= 0.1]
c2 = c2[c2['Z'] <= 1.1]
c2 = c2[(c2['X'] - 0.44)**2 + c2['Y']**2 <= (0.2)**2].drop_duplicates() 

# not_sus = df[['X', 'Y', 'Z']]
# not_sus = not_sus[not_sus['Z'] >= 1.2]
# not_sus = not_sus[not_sus['Z'] <= 0]
# sus1 = df[~df.index.isin(not_sus.index)][['X', 'Y', 'Z']]
sus1 = df[['X', 'Y', 'Z']]
sus1 = pd.concat([sus1[sus1['Z'] == 1.2], sus1[sus1['Z'] == 0]])
sus1 = sus1[(sus1['X']**2 + sus1['Y']**2) <= (1.5/2)**2]

sus2 = df[['X', 'Y', 'Z']]
sus2 = sus2[(sus2['X']**2 + sus2['Y']**2) == (1.5/2)**2]
sus = pd.concat([sus1,sus2]).drop_duplicates() 

void = df[~df.index.isin(c1.index)]
void = void[~void.index.isin(c2.index)].drop_duplicates() 

#c1_3D = pd.DataFrame({'X': c1[:,0],'Y': c1[:,1],'Z': c1[:,2]})
#c2_3D = pd.DataFrame({'X': c2[:,0],'Y': c2[:,1],'Z': c2[:,2]})
# c1.to_csv('c1_3D.csv')
# c2.to_csv('c2_3D.csv')
# sus.to_csv('sus_3D.csv')

In [38]:
margin = 0.1

c1_core = c1[c1['X']**2 + c1['Y']**2 <= 0.1**2]
c1_core = c1_core[c1_core['Z'] <= 0.8]
c1_core = c1_core[c1_core['Z'] >= 0.4].drop_duplicates() 

c2_core = c2[(c2['X'] - 0.44)**2 + c2['Y']**2 <= (0.1)**2]
c2_core = c2_core[c2_core['Z'] >= 0.4]
c2_core = c2_core[c2_core['Z'] <= 0.8].drop_duplicates() 

c1_surf1 = c1[c1['X']**2 + c1['Y']**2 <= (0.2*(1+margin))**2]
c1_surf1 = c1_surf1[c1_surf1['X']**2 + c1_surf1['Y']**2 >= (0.2*(1-margin))**2]
c1_surf2 = c1[c1['Z'] == 0.1]
c1_surf3 = c1[c1['Z'] == 1.1]
c1_surf = pd.concat([c1_surf1, c1_surf2, c1_surf3]).drop_duplicates()    

c2_surf1 = c2[(c2['X'] - 0.44)**2 + c2['Y']**2 <= (0.2*(1+margin))**2]
c2_surf1 = c2_surf1[(c2_surf1['X'] - 0.44)**2 + c2_surf1['Y']**2 >= (0.2*(1-margin))**2]
c2_surf2 = c2[c2['Z'] == 0.1]
c2_surf3 = c2[c2['Z'] == 1.1]
c2_surf = pd.concat([c2_surf1, c2_surf2, c2_surf3]).drop_duplicates()   


In [45]:
folder = 'data/nodes/'
c1.to_csv(folder+'c1.csv')
c2.to_csv(folder+'c2.csv')
sus.to_csv(folder+'sus.csv')
c1_core.to_csv(folder+'c1_core.csv')
c1_surf.to_csv(folder+'c1_surf.csv')
c2_core.to_csv(folder+'c2_core.csv')
c2_surf.to_csv(folder+'c2_surf.csv')


In [50]:
shape = 0
# excl = ['c1.csv', 'c2.csv']
for f in os.listdir(folder):
    shape_=pd.read_csv(folder+f).shape[0]
    print(shape_)
    shape+=shape_
shape

1451
2261
332


4044

In [41]:
import plotly.graph_objects as go
import numpy as np

def plot_point_cloud(nodes_list, color_list=None, name_list = None, base_list = None, cmap='Viridis', title='3D Point Cloud', filename='point_cloud.html', show_plot=False):
    """
    nodes: np array (N, 3) — columns: [R, angle, Z]
    color: None, a single color string, or an array of length N for colormap
    """
    if base_list is None:
        base_list = ['cyl' for _ in nodes_list]
    X = []
    Y = []
    Z = []
    markers = []
    fig = go.Figure()
    for i, nodes in enumerate(nodes_list):
            
        R = nodes[:, 0]
        theta = nodes[:, 1]  # radians, use np.deg2rad(nodes[:,1]) if degrees
        Z.append(nodes[:, 2])
        if base_list[i] == 'cyl':
            X.append(R * np.cos(theta))
            Y.append(R * np.sin(theta))
        elif base_list[i] == 'xyz':
            X.append(R)
            Y.append(theta)
        else:
            raise
        marker = dict(size=3, colorscale=cmap, showscale=True)

        if color_list is None:
            marker['color'] = 'steelblue'
            marker['showscale'] = False
        elif isinstance(color_list, list):
            marker['color'] = color_list[i]
            marker['showscale'] = False
        else:
            marker['color'] = color_list[i]
        markers.append(marker)
        if not name_list is None:
            name = name_list[i]
        else:
            name = f'Series {i}'


        fig.add_trace(go.Scatter3d(
            x=X[i], y=Y[i], z=Z[i],
            mode='markers',
            marker=marker,
            name=name                ))

    fig.update_layout(
    title=title,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'  # preserves true proportions
    )
)


    if show_plot:
        fig.show()  # also shows inline in notebook
    else:
        fig.write_html(filename)
        print(f"Saved to {filename}")

# no color
plot_point_cloud([c1.to_numpy(), c2.to_numpy(), sus.to_numpy(), c1_core.to_numpy(), c1_surf.to_numpy(), c2_core.to_numpy(), c2_surf.to_numpy()], ['red', 'blue', 'green', 'tomato', 'steelblue', 'tomato', 'steelblue'], ['C1', 'C2','Sus', 'C1 core', 'C1 surf', 'C2 core', 'C2 surf'], base_list=['xyz', 'xyz', 'xyz', 'xyz', 'xyz', 'xyz', 'xyz'], filename='plots/3Dtemps.html', show_plot=True)